In [ ]:
# Treino CNN - ResNet50, EfficientNet-B0, ConvNeXt-Tiny e DenseNet121 (P2 · Tema F)
# Fine-tuning de quatro CNN pré-treinadas (ImageNet) no FieldVitis, com o `splits.json` do repositório (3 seeds).

In [ ]:
# --- Configuração: dados (Drive) + splits (GitHub) ---
from google.colab import drive
drive.mount("/content/drive")

!git clone -q https://github.com/franciscorafael-dotcom/DL_P2_TP2_G2-F1.git
!mkdir -p /content/fieldvitis && unzip -q -o /content/drive/MyDrive/fieldvitis.zip -d /content/fieldvitis/

import json
from pathlib import Path
DATA_DIR = Path("/content/fieldvitis")
REPO_DIR = Path("/content/DL_P2_TP2_G2-F1")

# Verificação: as imagens batem com o splits.json?
sj = json.load(open(REPO_DIR / "data/splits.json"))
CLASSES = sj["classes"]
print("Pastas:      ", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))
print("splits.json: ", CLASSES)
for s in ["0", "1", "2"]:
    sp = sj["seeds"][s]
    faltam = [p for k in sp for p in sp[k] if not (DATA_DIR / p).exists()]
    print(f"seed {s}:", {k: len(v) for k, v in sp.items()}, "| em falta:", len(faltam))

: 

In [ ]:
# Configuração e Carregamento das imagens
import time, gc
import numpy as np, pandas as pd
import tensorflow as tf, keras
from keras import layers
from PIL import Image, ImageOps
from sklearn.metrics import f1_score, recall_score

IMG_SIZE, BATCH = 224, 32
EPOCHS_HEAD, LR_HEAD = 5, 1e-3     # fase 1: só a cabeça (base congelada)
EPOCHS_FT, LR_FT = 25, 1e-5        # fase 2: fine-tuning da rede toda
PATIENCE = 6                       # early stopping no Macro-F1 de validação
SEEDS = [0, 1, 2]
RUNS_DIR = Path("/content/drive/MyDrive/P2_runs"); RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Keras", keras.__version__, "| backend", keras.backend.backend(),
      "| GPU:", tf.config.list_physical_devices("GPU"))

# Todas as imagens carregadas uma vez: RGB, orientação EXIF corrigida, 224x224
all_paths = sorted({p for s in sj["seeds"].values() for k in ("train", "val", "test") for p in s[k]})
cls_idx = {c: i for i, c in enumerate(CLASSES)}
IMGS, LABELS = {}, {}
for p in all_paths:
    im = ImageOps.exif_transpose(Image.open(DATA_DIR / p)).convert("RGB")
    IMGS[p] = np.asarray(im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)
    LABELS[p] = cls_idx[Path(p).parts[0]]
print("Imagens carregadas:", len(IMGS), "| por classe:", np.bincount(list(LABELS.values())))

In [ ]:
# Cada rede espera os píxeis num formato diferente (o mesmo usado no treino do ImageNet)
PREPROCESS = {
    "resnet50": keras.applications.resnet50.preprocess_input,           # converte para BGR e subtrai a média do ImageNet
    "efficientnetb0": keras.applications.efficientnet.preprocess_input,  # não altera: o modelo normaliza por dentro
    "convnext_tiny": keras.applications.convnext.preprocess_input,       # não altera: o modelo normaliza por dentro
    "densenet121": keras.applications.densenet.preprocess_input,         # escala para [0,1] e normaliza com média/desvio do ImageNet
}

def preparar_dados(seed, nome):
    """Devolve treino/val/teste da seed indicada, já pré-processados para o modelo."""
    sp = sj["seeds"][str(seed)]
    dados = {}
    for split in ["train", "val", "test"]:
        x = np.stack([IMGS[p] for p in sp[split]]).astype("float32")
        y = np.array([LABELS[p] for p in sp[split]])
        dados[split] = (PREPROCESS[nome](x), y, sp[split])
    return dados

In [ ]:
def criar_modelo(nome):
    """Base pré-treinada no ImageNet + augmentation + camada final de 4 classes."""
    if nome == "resnet50":
        base = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                           input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    elif nome == "efficientnetb0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                                 input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    elif nome == "convnext_tiny":
        base = keras.applications.ConvNeXtTiny(include_top=False, weights="imagenet",
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    else:
        base = keras.applications.DenseNet121(include_top=False, weights="imagenet",
                                              input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    base.trainable = False   # começa congelada

    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.2),
    ])  # só atua durante o treino

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    x = base(x, training=False)   # BatchNorm fica em modo inferência, mesmo no fine-tuning
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
    return keras.Model(inputs, outputs), base

In [ ]:
# Treinar e Guardar
def guardar_previsoes(nome, seed, split, caminhos, y, probs):
    """CSV para o avaliador: uma linha por imagem, com a classe verdadeira, a prevista e as 4 probabilidades."""
    df = pd.DataFrame(probs, columns=[f"p_{c}" for c in CLASSES])
    df.insert(0, "y_pred", [CLASSES[i] for i in probs.argmax(axis=1)])
    df.insert(0, "y_true", [CLASSES[i] for i in y])
    df.insert(0, "path", caminhos)
    df.insert(0, "split", split)
    df.insert(0, "seed", seed)
    df.insert(0, "modelo", nome)
    df.to_csv(RUNS_DIR / f"preds_{nome}_seed{seed}_{split}.csv", index=False)

class EarlyStoppingMacroF1(keras.callbacks.Callback):
    """Early stopping no Macro-F1 de validação (maximizar); restaura os melhores pesos no fim."""
    def __init__(self, x_val, y_val, patience):
        super().__init__()
        self.x_val, self.y_val, self.patience = x_val, y_val, patience

    def on_train_begin(self, logs=None):
        self.best, self.wait, self.best_weights = -1.0, 0, None

    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.x_val, batch_size=BATCH, verbose=0).argmax(axis=1)
        f1 = f1_score(self.y_val, pred, average="macro")
        print(f"    val_macro_f1 = {f1:.4f} (melhor: {max(self.best, f1):.4f})")
        if f1 > self.best:
            self.best, self.wait = f1, 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

def treinar(nome, seed):
    keras.utils.set_random_seed(seed)   # reprodutibilidade
    dados = preparar_dados(seed, nome)
    x_tr, y_tr, _ = dados["train"]
    x_va, y_va, _ = dados["val"]

    # Pesos por classe: o black rot (poucas imagens) pesa mais na loss
    contagens = np.bincount(y_tr, minlength=len(CLASSES))
    class_weight = {i: len(y_tr) / (len(CLASSES) * n) for i, n in enumerate(contagens)}

    model, base = criar_modelo(nome)

    # Fase 1: treina só a camada final (base congelada)
    model.compile(optimizer=keras.optimizers.Adam(LR_HEAD),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_HEAD,
              batch_size=BATCH, class_weight=class_weight, verbose=2)

    # Fase 2: fine-tuning de toda a rede, com learning rate baixo
    base.trainable = True
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(LR_FT),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    early = EarlyStoppingMacroF1(x_va, y_va, PATIENCE)
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_FT,
              batch_size=BATCH, class_weight=class_weight, callbacks=[early], verbose=2)

    # Guardar o modelo (só no Drive) e as previsões de val e teste (para o avaliador)
    model.save(RUNS_DIR / f"{nome}_seed{seed}.keras")
    for split in ["val", "test"]:
        x, y, caminhos = dados[split]
        guardar_previsoes(nome, seed, split, caminhos, y, model.predict(x, verbose=0))

    _, val_acc = model.evaluate(x_va, y_va, verbose=0)
    print(f">>> {nome} seed {seed}: accuracy de validação = {val_acc:.3f}")

In [ ]:
# ResNet50
for seed in SEEDS:
    print(f"\n===== ResNet50 | seed {seed} =====")
    treinar("resnet50", seed)
    keras.backend.clear_session()   # liberta a memória da GPU entre corridas

In [ ]:
# EfficientNet-B0
for seed in SEEDS:
    print(f"\n===== EfficientNet-B0 | seed {seed} =====")
    treinar("efficientnetb0", seed)
    keras.backend.clear_session()

In [ ]:
# ConvNeXt-Tiny
for seed in SEEDS:
    print(f"\n===== ConvNeXt-Tiny | seed {seed} =====")
    treinar("convnext_tiny", seed)
    keras.backend.clear_session()


In [ ]:
# DenseNet121
for seed in SEEDS:
    print(f"\n===== DenseNet121 | seed {seed} =====")
    treinar("densenet121", seed)
    keras.backend.clear_session()


In [ ]:
import glob
for f in sorted(glob.glob("/content/drive/MyDrive/P2_runs/preds_*_val.csv")):
    df = pd.read_csv(f)
    print(f"{f.split('/')[-1]:40s} accuracy val = {(df.y_true == df.y_pred).mean():.3f}")

In [ ]:
# Métricas por seed e média das 3 seeds (conjunto de teste)
from sklearn.metrics import accuracy_score
linhas = []
for f in sorted(glob.glob(str(RUNS_DIR / "preds_*_test.csv"))):
    df = pd.read_csv(f)
    linhas.append({"modelo": df.modelo[0], "seed": df.seed[0],
                   "accuracy": accuracy_score(df.y_true, df.y_pred),
                   "macro_f1": f1_score(df.y_true, df.y_pred, average="macro"),
                   "macro_recall": recall_score(df.y_true, df.y_pred, average="macro")})
por_seed = pd.DataFrame(linhas)
print("Métricas por seed:")
print(por_seed.round(4).to_string(index=False))
print("\nMédia das 3 seeds por modelo:")
print(por_seed.groupby("modelo")[["accuracy", "macro_f1", "macro_recall"]].mean().round(4))
